In [10]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.naive_bayes import MultinomialNB

In [11]:
# Check if required files exist
import os

required_files = ['spam_train.csv', 'spam_test.csv']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file '{file}' is missing. Please ensure it is in the same folder as this notebook.")

# Instructions for Running This Notebook
Please ensure that the following files are in the same folder as this notebook:
- spam_train.csv
- spam_test.csv

To run the notebook:
1. Extract all files into the same folder.
2. Open `yazdanpanah-asal-610002071-project` in Jupyter Notebook.
3. Run all cells.

# Approach Explanation
This notebook implements a spam classification problem using the Naive Bayes algorithm. The key steps are as follows:

1. **Data Preprocessing:**
   - Text data is cleaned by converting to lowercase and removing punctuation and special characters.
   - This ensures consistency and reduces noise in the data.

2. **Bag-of-Words Representation:**
   - Text data is transformed into a numerical format using the Bag-of-Words (BoW) model.
   - The vocabulary is built from the training data and is used consistently for both training and testing datasets.

3. **Manual Implementation of Naive Bayes:**
   - The Naive Bayes model is implemented from scratch with Laplace smoothing to handle unseen words in the test data.
   - Log transformations are used to prevent numerical underflow when calculating probabilities.

4. **Scikit-learn Implementation:**
   - Scikit-learn's `MultinomialNB` is used as a baseline to compare with the manual implementation.
   - Both models are evaluated using metrics such as Accuracy, Precision, Recall, and F1-Score.

5. **Comparison:**
   - The performance of the manual implementation is compared with Scikit-learn's implementation to highlight similarities and differences in results.

In [ ]:
# Load the training and testing datasets
train_data = pd.read_csv('spam_train.csv')
test_data = pd.read_csv('spam_test.csv')

# Function to preprocess the text
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = ''.join([char for char in text if char.isalnum() or char.isspace()])
    return text

# Apply preprocessing to the training and testing data
train_data['message'] = train_data['message'].apply(preprocess_text)
test_data['message'] = test_data['message'].apply(preprocess_text)

# Separate features (X) and labels (y) for training and testing data
X_train = train_data['message']
y_train = train_data['label']
X_test = test_data['message']
y_test = test_data['label']

# Display the first few rows of the training data
print("First 5 rows of Training Data:")
print(train_data.head())

In [ ]:
# Create Bag-of-Words (BoW) representation using CountVectorizer
vectorizer = CountVectorizer()

# Fit the vectorizer on the training data and transform it
X_train_bow = vectorizer.fit_transform(X_train)

# Transform the test data using the same vocabulary
X_test_bow = vectorizer.transform(X_test)

# Display the size of the vocabulary
print("Vocabulary Size:", len(vectorizer.vocabulary_))

# Show a sample of the feature names (first 10 words in the vocabulary)
print("\nSample Vocabulary Words:", list(vectorizer.vocabulary_.keys())[:10])

In [14]:
# Define the Naive Bayes class
class NaiveBayes:
    def __init__(self):
        self.prior = {}          # To store P(c) for each class
        self.likelihood = {}     # To store P(w|c) for each word and class
        self.vocabulary = set()  # To store the vocabulary
        self.class_counts = {}   # To store the total word counts for each class

    def fit(self, X, y, alpha=1):
        """
        Train the Naive Bayes model with Laplace smoothing.
        """
        # Initialize variables
        n_samples, n_features = X.shape
        self.vocabulary = set(range(n_features))  # Vocabulary is column indices

        # Step 1: Calculate prior probabilities P(c) with additive smoothing
        unique_classes, class_counts = np.unique(y, return_counts=True)
        total_classes = len(unique_classes)  # Total number of classes (e.g., spam, ham)
        self.class_counts = dict(zip(unique_classes, class_counts))  # Store class counts
        self.prior = {c: (class_counts[i] + alpha) / (n_samples + alpha * total_classes)
                      for i, c in enumerate(unique_classes)}  # Additive smoothing for P(c)

        # Step 2: Calculate likelihood P(w|c) with Laplace smoothing
        self.likelihood = {c: np.ones(n_features) * alpha for c in unique_classes}  # Initialize with alpha
        for c in unique_classes:
            # Sum word occurrences for each feature in class c
            class_indices = np.where(y == c)[0]
            self.likelihood[c] += X[class_indices].sum(axis=0).A1  # Add word counts
            # Normalize likelihood
            total_count = self.likelihood[c].sum()
            self.likelihood[c] /= total_count

    def predict(self, X):
        """
        Predict the class labels for input X.
        """
        # Calculate log probabilities to prevent underflow
        log_prior = {c: np.log(p) for c, p in self.prior.items()}
        log_likelihood = {c: np.log(self.likelihood[c]) for c in self.likelihood}

        predictions = []
        for i in range(X.shape[0]):
            # Calculate log P(c|X) for each class
            log_posterior = {}
            for c in self.prior:
                log_posterior[c] = log_prior[c] + (X[i].toarray() * log_likelihood[c]).sum()
            # Choose the class with the highest posterior probability
            predictions.append(max(log_posterior, key=log_posterior.get))
        return np.array(predictions)

In [ ]:
# Initialize and train the manual Naive Bayes model
nb_model = NaiveBayes()
nb_model.fit(X_train_bow, y_train)

# Predict on the test dataset
y_pred_manual = nb_model.predict(X_test_bow)

# Evaluate the manual implementation
manual_accuracy = accuracy_score(y_test, y_pred_manual)
manual_precision = precision_score(y_test, y_pred_manual)
manual_recall = recall_score(y_test, y_pred_manual)
manual_f1 = f1_score(y_test, y_pred_manual)

# Print performance metrics
print("Manual Naive Bayes Implementation:")
print(f"Accuracy: {manual_accuracy:.2f}")
print(f"Precision: {manual_precision:.2f}")
print(f"Recall: {manual_recall:.2f}")
print(f"F1-Score: {manual_f1:.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_manual))

In [ ]:
# Train a Naive Bayes model using Scikit-learn
sklearn_model = MultinomialNB()
sklearn_model.fit(X_train_bow, y_train)

# Predict on the test dataset
y_pred_sklearn = sklearn_model.predict(X_test_bow)

# Evaluate the Scikit-learn implementation
sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)
sklearn_precision = precision_score(y_test, y_pred_sklearn)
sklearn_recall = recall_score(y_test, y_pred_sklearn)
sklearn_f1 = f1_score(y_test, y_pred_sklearn)

# Print performance metrics
print("Scikit-learn Naive Bayes Implementation:")
print(f"Accuracy: {sklearn_accuracy:.2f}")
print(f"Precision: {sklearn_precision:.2f}")
print(f"Recall: {sklearn_recall:.2f}")
print(f"F1-Score: {sklearn_f1:.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_sklearn))

In [ ]:
# Compare the performance of the manual and Scikit-learn implementations
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Manual Naive Bayes": [manual_accuracy, manual_precision, manual_recall, manual_f1],
    "Scikit-learn Naive Bayes": [sklearn_accuracy, sklearn_precision, sklearn_recall, sklearn_f1]
})

# Display the comparison table in a readable format
print("Comparison of Manual and Scikit-learn Naive Bayes Models:\n")
print(comparison.to_string(index=False))

# **Observations and Insights**

1. **Performance Comparison**:
   - Both the manual and Scikit-learn implementations achieved identical performance metrics (Accuracy, Precision, Recall, and F1-Score). This confirms the correctness of the manual implementation.
   - The metrics reflect that the model performs well on the spam classification task.

2. **Advantages of Scikit-learn Implementation**:
   - Scikit-learn's `MultinomialNB` is highly optimized for speed and scalability, making it ideal for large datasets.
   - It requires no manual computation of probabilities, reducing potential for human error.

3. **Value of Manual Implementation**:
   - Implementing the algorithm manually provides a deeper understanding of concepts like prior probabilities, likelihoods, Laplace smoothing, and log transformations.
   - It demonstrates the inner workings of the Naive Bayes algorithm, which is often abstracted away in library implementations.

In [ ]:
import matplotlib.pyplot as plt

# Define metrics and their corresponding values
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
manual_values = [manual_accuracy, manual_precision, manual_recall, manual_f1]
sklearn_values = [sklearn_accuracy, sklearn_precision, sklearn_recall, sklearn_f1]

# Bar chart
x = np.arange(len(metrics))
width = 0.35  # Bar width

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, manual_values, width, label="Manual Naive Bayes", color="skyblue")
plt.bar(x + width/2, sklearn_values, width, label="Scikit-learn Naive Bayes", color="orange")

# Add labels, title, and legend
plt.xticks(x, metrics)
plt.ylabel("Scores")
plt.title("Comparison of Manual and Scikit-learn Naive Bayes Metrics")
plt.legend(loc="lower right")

# Display the plot
plt.tight_layout()
plt.show()